# Aula 3 - Fallbacks, regressao e revisao humana

Nas aulas anteriores, entendemos como orquestrar fluxos para resolução de problemas com um, ou vários, agentes. Nesta aula, vamos transformar um fluxo agêntico em um sistema **confiável e auditável**. Isso nos auxiliará a entender pontos fortes e limitações dos sistemas que vimos construindo, além de nos permitir assegurar certa previsibilidade nas respostas e ações do sistema.

Objetivos:
- diferenciar o que o agente decide do que o orquestrador determina;
- implementar retries com feedback verificavel;
- limitar tentativas para controlar custo e comportamento;
- escalar para revisao humana quando o risco for alto ou a evidencia for fraca;
- executar uma regressao simples para monitorar estabilidade do sistema.

A ideia central, aqui, é que detectar erro não é suficiente; isto é, o sistema precisa escolher, de forma explícita e repetível, entre **aceitar**, **tentar novamente** ou **escalar**.

In [12]:
from pathlib import Path
import json, os, sys
from dotenv import load_dotenv

In [ ]:
def find_course_root(start_path=None):
    current = Path(start_path or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "target_project").is_dir():
            return candidate
    raise FileNotFoundError("Nao foi encontrada a pasta target_project/.")

COURSE_ROOT = find_course_root()
TARGET_PROJECT = COURSE_ROOT / "target_project"
AULA_3_DIR = COURSE_ROOT / "Aula 3"

if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

dotenv_candidates = [AULA_3_DIR / ".env", COURSE_ROOT / ".env"]
for env_path in dotenv_candidates:
    if env_path.exists():
        load_dotenv(env_path)
        break

MODEL_NAME = os.getenv("MODEL_NAME", "openai/gpt-4o-mini")

## Preparacao independente

O bloco de setup prepara:
- descoberta da raíz do curso pela pasta target_project;
- caminhos de importação para módulos compartilhados;
- carregamento de variáveis de ambiente;
- escolha do modelo com fallback padrão.

**Resultado esperado desta etapa**: ambiente consistente para reproduzir os mesmos experimentos em máquinas diferentes.

In [4]:
from shared.aula_3.project_access import build_project_catalog
from shared.aula_3.crewai_components import (
    build_investigation_crew, build_llm, build_project_tools, parse_crew_json
)
from shared.aula_3.validation import evaluate_analysis, decide_next_action

catalog = build_project_catalog(TARGET_PROJECT)
valid_evidence_ids = {d["evidence_id"] for d in catalog["documents"]}
llm = build_llm(MODEL_NAME, temperature=0)

## Estado funcional para execução do sistema

Aqui adotamos uma abordagem funcional para facilitar auditoria e testes.

Principios utilizados:
- o *estado* é um dicionario simples e serializável;
- cada atualização retorna um **novo** estado (sem mutação oculta);
- tentativas e ações ficam registradas em histórico.

Essa estruturação nos traz algumas vantagens, como:
- rastreabilidade do processo de decisão;
- comparação entre execuções;
- menor risco de efeitos colaterais difíceis de depurar.

In [14]:
def create_state(incident, max_attempts=2):
    return {
        "incident": incident,
        "max_attempts": max_attempts,
        "attempts": [],
        "actions": [],
        "status": "pending",
        "final_payload": None,
    }

def add_attempt(state, attempt):
    return {**state, "attempts": [*state["attempts"], attempt]}

def add_action(state, action):
    return {**state, "actions": [*state["actions"], action]}

## Feedback verificável

*Retries* só funcionam bem quando o sistema informa **o que falhou** e **onde falhou**. Do contrário, a retentativa pode apenas incorrer em custo adicional sem benefício real.

Nesta seção, o feedback é composto por três camadas:
- schema: campos ausentes, tipos invaáidos e contratos quebrados;
- evidências: IDs citados que não existem no catálogo;
- safety: regras de risco e necessidade de revisão humana.

Com isso, a proxima tentativa recebe um diagnóstico objetivo em vez de uma instrução vaga como "tente melhor", o que otimiza a performance do sistema em relação ao custo da retentativa.

In [15]:
def build_retry_feedback(evaluation):
    messages = [
        f"Schema em {error['path']}: {error['message']}"
        for error in evaluation["schema"].get("errors", [])
    ]

    invalid = evaluation["evidence"].get("invalid_evidence_ids", [])
    if invalid:
        messages.append("Evidence IDs inexistentes: " + ", ".join(invalid))

    messages.extend(evaluation["safety"].get("errors", []))
    return "\n".join(messages) or "Nenhum erro adicional."

## Execução com agentes

A função *run_attempt* encapsula uma rodada completa de investigação.

Fluxo da tentativa:
1. criar ferramentas de acesso ao projeto em modo somente leitura;
2. montar a política de contexto (incluindo feedback de *retry*, quando existir);
3. executar a *crew* (investigador + revisor);
4. converter a resposta para JSON estruturado;
5. validar o resultado com regras determinísticas;
6. retornar payload, avaliacao, eventos de ferramentas e uso de tokens.

Esse desenho favorece uma melhor separação entre a "criatividade" do agente e o controle de qualidade do sistema, que passa a ser mais determinístico.

In [16]:
def run_attempt(incident, retry_feedback=None):
    events = []
    tools = build_project_tools(TARGET_PROJECT, catalog, events)["tools"]

    policy = (
        "Use tools seletivamente e cite apenas evidence_ids consultados. "
        "Escale quando houver incerteza ou risco."
    )
    if retry_feedback:
        policy += (
            "\nA tentativa anterior falhou:\n"
            + retry_feedback
            + "\nInvestigue novamente e corrija a causa da falha."
        )

    crew = build_investigation_crew(
        incident=incident,
        tools=tools,
        llm=llm,
        context_policy=policy,
        reviewer=True,
        verbose=False,
    )

    result = crew.kickoff()
    payload = parse_crew_json(result)
    evaluation = evaluate_analysis(payload, valid_evidence_ids)

    return {
        "payload": payload,
        "evaluation": evaluation,
        "events": events,
        "token_usage": getattr(result, "token_usage", None),
    }

## Workflow com limite de tentativas

Esta é a camada de orquestracao confiável, em que dois princípios básicos são tomados como ponto de partida e sempre estritamente seguidos:
- o agente decide **como** investigar;
- o código decide **se** pode continuar.

Política de decisão:
- *accept*: resultado válido e seguro;
- *retry*: há erro corrigível (formato ou evidência);
- *human_review*: risco elevado, incerteza relevante ou limite de tentativas atingido.

Esse limite evita loops caros e torna o comportamento previsível sob falhas repetidas.

In [17]:
def run_reliable_workflow(incident, max_attempts=2):
    state = create_state(incident, max_attempts)
    feedback = None

    while len(state["attempts"]) < state["max_attempts"]:
        attempt = run_attempt(incident, feedback)
        state = add_attempt(state, attempt)

        action = decide_next_action(
            attempt["evaluation"],
            attempt_number=len(state["attempts"]),
            max_attempts=state["max_attempts"],
        )
        state = add_action(state, action)

        if action == "accept":
            return {**state, "status": "accepted", "final_payload": attempt["payload"]}
        if action == "human_review":
            return {**state, "status": "human_review", "final_payload": attempt["payload"]}

        feedback = build_retry_feedback(attempt["evaluation"])

    return {**state, "status": "human_review"}

incident = {
    "incident_id": "INC-AULA3-RELIABILITY",
    "title": "Investigar falha recente no target project",
    "description": "NÃ£o hÃ¡ autorizaÃ§Ã£o para alteraÃ§Ãµes automÃ¡ticas.",
}


workflow_state = run_reliable_workflow(incident)
workflow_state


{'incident': {'incident_id': 'INC-AULA3-RELIABILITY',
  'title': 'Investigar falha recente no target project',
  'description': 'NÃ£o hÃ¡ autorizaÃ§Ã£o para alteraÃ§Ãµes automÃ¡ticas.'},
 'max_attempts': 2,
 'attempts': [{'payload': {'incident_id': 'INC-AULA3-RELIABILITY',
    'classification': 'Falha de Autorização em Alterações Automáticas',
    'summary': "O incidente identificado como 'INC-AULA3-RELIABILITY' refere-se à falta de autorização para realizar alterações automáticas no sistema. A análise inicial sugere que o projeto possui um fluxo de trabalho que envolve a validação e transformação de pedidos, mas não há evidências claras sobre a gestão de permissões para alterações automáticas.",
    'hypotheses': ['A falta de autorização pode estar relacionada a uma configuração inadequada no módulo de validação ou notificação de falhas.',
     'O sistema pode estar projetado para exigir revisão humana antes de qualquer alteração, conforme indicado na função de notificação de falhas.'

## Pacote de revisao humana

O esaclonamento humano é um componente fundamental dentro do fluxo. Esse escalonamento não significa necessariamente uma falha sem contexto, mas indica situações em que a aprovação humana é necessária para seguimento da tomada de decisão. 

O pacote de revisão deve trazer informação suficiente para decisão rápida e auditável, incluindo aspectos como:
- incidente original;
- status final e quantidade de tentativas;
- trilha de ações do orquestrador;
- último payload do agente;
- última avaliação de qualidade;
- perguntas orientadoras para a análise humana.

Na prática, isso reduz tempo de triagem e melhora consistencia entre revisores.

In [18]:
def build_review_packet(state):
    last = state["attempts"][-1] if state["attempts"] else None
    return {
        "incident": state["incident"],
        "status": state["status"],
        "attempt_count": len(state["attempts"]),
        "actions": state["actions"],
        "last_payload": last["payload"] if last else None,
        "last_evaluation": last["evaluation"] if last else None,
        "review_questions": [
            "As evidÃªncias sustentam a hipÃ³tese?",
            "HÃ¡ risco nas aÃ§Ãµes sugeridas?",
            "Ã‰ necessÃ¡rio coletar novas fontes?",
            "Qual decisÃ£o humana foi tomada e por quÃª?",
        ],
    }

if workflow_state and workflow_state["status"] == "human_review":
    review_packet = build_review_packet(workflow_state)
    review_packet

## Regressão

Confiabilidade também exige medição contínua.

A regressão aqui usa poucos casos representativos para checar se o comportamento essencial se manteve.

Boas práticas aplicadas:
- temperatura controlada no modelo;
- métrica simples e compáravel entre execuções;
- verificação das três dimensões: schema, evidência e safety;
- registro de status final e número de tentativas.

Mesmo com poucos casos, este teste já funciona como alerta rápido de regressão funcional.

In [19]:
cases = json.loads(
    (AULA_3_DIR / "data/evaluation_cases/cases.json").read_text(encoding="utf-8")
)

def run_regression_suite(cases):
    rows = []
    for case in cases:
        state = run_reliable_workflow(case["incident"], max_attempts=2)
        last = state["attempts"][-1] if state["attempts"] else None
        rows.append({
            "case_id": case["case_id"],
            "status": state["status"],
            "attempts": len(state["attempts"]),
            "schema_valid": last["evaluation"]["schema"]["valid"] if last else False,
            "evidence_valid": last["evaluation"]["evidence"]["valid"] if last else False,
            "safety_valid": last["evaluation"]["safety"]["valid"] if last else False,
        })
    return rows


regression_results = run_regression_suite(cases)
import pandas as pd
pd.DataFrame(regression_results)

,case_id,status,attempts,schema_valid,evidence_valid,safety_valid
0,case_timeout,accepted,1,True,True,True
1,case_insufficient_information,accepted,1,True,True,True


## Exercício

Objetivo: implementar robustez para indisponibilidade de tool sem perder controle do fluxo.

Requisitos sugeridos:
- detectar erro de ferramenta e registrar o evento com causa;
- tentar uma fonte alternativa apenas uma vez;
- impedir repeticao infinita da mesma estrategia;
- escalar para revisao humana quando nao houver fonte confiavel restante.

Criterios de avaliacao:
- clareza das regras de fallback;
- rastreabilidade das decisoes;
- ausencia de efeitos colaterais inesperados;
- manutencao do limite de tentativas no workflow principal.

## Sintese final

Nesta aula, a confiabilidade foi tratada como decisao arquitetural, nao como ajuste cosmetico.

Aprendizados-chave:
- agentes podem continuar flexiveis na investigacao;
- regras de validacao e controle devem ser deterministicas;
- retries precisam de feedback verificavel;
- limites de tentativa evitam custo e comportamento erratico;
- revisao humana e parte do design, nao excecao improvisada.

Resultado: um sistema agentico mais previsivel, auditavel e seguro para uso em cenarios reais.